# Transmitancia de las 61 moléculas de HITRAN en 8–14 µm

Para cada molécula del catálogo de pylblrtm (`MOLECULE_IDS`, los mismos IDs que HITRAN/HAPI):

1. se piden sus líneas a HITRAN (edición HITRAN2024) con **todos sus isotopólogos** (`iso_id="all"`),
2. se calcula la transmitancia en **8–14 µm (714.29–1250 cm⁻¹) con paso de 0.1 cm⁻¹**,
3. se grafica.

Si una molécula no se puede calcular, en su sección se imprime la **excepción y el porqué** y el notebook sigue con la siguiente.

**Escenario:** un camino standoff homogéneo (`HomogeneousPath`) con la concentración, longitud, presión y temperatura de la celda de configuración (por defecto 1 ppm en 100 m, 1013.25 hPa, 296 K). Se puede cambiar ahí y volver a correr todo.

**Abundancias:** las intensidades de HITRAN ya incluyen la abundancia natural de cada isotopólogo, así que la concentración es la **total de la molécula**; no hay que multiplicar por la abundancia.

**Sobre el paso de 0.1 cm⁻¹:** es la transmitancia monocromática evaluada cada 0.1 cm⁻¹ (no un espectro convolucionado con un instrumento). A 1 atm las líneas tienen ~0.07 cm⁻¹ de semiancho, así que los picos se ven pero su profundidad exacta puede quedar entre dos puntos.

In [ ]:
import contextlib
import io
import logging
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import pylblrtm
from pylblrtm import (HomogeneousPath, MOLECULE_IDS, from_hapi, get_abundance, run_forward,
                      HITRANDataError, HITRANNoLinesError, HITRANNotAvailableError,
                      HITRANQuotaError)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# ── Configuración ─────────────────────────────────────────────────────────────
UM_MIN, UM_MAX = 8.0, 14.0
NU_MIN, NU_MAX = 1e4 / UM_MAX, 1e4 / UM_MIN      # 714.286–1250 cm⁻¹
STEP_CM1 = 0.1                                    # resolución pedida [cm⁻¹]

PPM = 1.0                # concentración de la molécula [ppm]
PATH_M = 100.0           # longitud del camino [m]
P_HPA = 1013.25          # presión [hPa]
T_K = 296.0              # temperatura [K]

# Una carpeta por molécula: HAPI recarga todas las tablas de la carpeta en cada
# descarga, así que una carpeta única se vuelve lenta
CACHE = Path.home() / ".pylblrtm" / "cache_8_14um"

try:                     # humlicek (numba) es mucho más rápido; wofz es la referencia
    import numba  # noqa: F401
    VOIGT = "humlicek"
except ImportError:
    VOIGT = "wofz"
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

print(f"pylblrtm {pylblrtm.__version__} · voigt={VOIGT} · n_jobs={N_JOBS}")
print(f"{NU_MIN:.3f}–{NU_MAX:.3f} cm⁻¹, paso {STEP_CM1} cm⁻¹ · "
      f"{PPM:g} ppm × {PATH_M:g} m, {P_HPA:g} hPa, {T_K:g} K · caché {CACHE}")

In [ ]:
# Estilo: una serie por figura (el título la nombra), tinta y rejilla recesivas
SERIES = "#2a78d6"
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

RESULTS = []   # una fila por molécula para el resumen final


def firma(mol):
    """Pide las líneas de *mol* (todos sus isotopólogos), calcula T y la grafica.

    Si no se puede, imprime la excepción y el porqué en vez de detener el notebook.
    """
    t0 = time.perf_counter()
    try:
        with contextlib.redirect_stdout(io.StringIO()):          # HAPI imprime mucho
            sd = from_hapi(mol, NU_MIN, NU_MAX, iso_id="all", cache_dir=CACHE / mol)
        path = HomogeneousPath(PATH_M, {mol: PPM * 1e-6}, pressure_hpa=P_HPA, temperature_k=T_K)
        cfg = {"spectral": {"nu_min_cm1": NU_MIN, "nu_max_cm1": NU_MAX, "step_cm1": STEP_CM1},
               "calculation": {"voigt": VOIGT, "n_jobs": N_JOBS},
               "output": {"transmittance": True, "optical_depth": True}}
        res = run_forward(sd, path, cfg)
        nu, T, tau = res.wavenumber_cm1, res.transmittance, res.optical_depth
    except HITRANNoLinesError as err:
        why = "HITRAN no tiene líneas de esta molécula en 8–14 µm (sí en otros rangos)."
        return _fail(mol, err, why, t0)
    except HITRANNotAvailableError as err:
        why = ("HITRANonline no la entrega línea por línea en ningún rango; existe solo como "
               "sección eficaz (https://hitran.org/xsc/), que pylblrtm todavía no lee.")
        return _fail(mol, err, why, t0)
    except HITRANQuotaError as err:
        why = ("HITRAN limita las consultas diarias a su API y hoy se superó el límite. "
               "Vuelve a correr la celda mañana: lo ya descargado queda en la caché.")
        return _fail(mol, err, why, t0)
    except HITRANDataError as err:
        return _fail(mol, err, "HITRAN/HAPI devolvió datos que la librería no puede usar.", t0)
    except Exception as err:   # inesperado: se muestra igual, para poder corregirlo
        return _fail(mol, err, "Error inesperado de la librería.", t0)

    isos = sorted(i for (_, i) in sd.tips)
    um = 1e4 / nu
    k = int(np.argmin(T))

    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.plot(um, T, color=SERIES, linewidth=1.5)
    ax.set_xlim(UM_MIN, UM_MAX)
    ax.set_ylim(min(0.0, T.min()) - 0.02, 1.02)
    ax.set_xlabel("Longitud de onda [µm]")
    ax.set_ylabel("Transmitancia")
    inv = lambda x: 1e4 / np.maximum(x, 1e-9)        # µm ↔ cm⁻¹
    top = ax.secondary_xaxis("top", functions=(inv, inv))
    top.set_xlabel("Número de onda [cm⁻¹]", color=INK_2)
    top.tick_params(colors=MUTED)
    ax.set_title(f"{mol} (mol_id {MOLECULE_IDS[mol]}) · {sd.n_lines:,} líneas · "
                 f"isotopólogos {isos} · {PPM:g} ppm × {PATH_M:g} m",
                 color=INK, loc="left", fontsize=11)
    if T.min() < 0.999:      # etiqueta solo el punto de máxima absorción
        ax.annotate(f"T mín {T.min():.3f}\n{um[k]:.3f} µm", xy=(um[k], T[k]),
                    xytext=(8, 8), textcoords="offset points", color=INK_2, fontsize=9)
    plt.show()

    print(f"✓ {mol}: {sd.n_lines:,} líneas de los isotopólogos {isos} "
          f"(abundancias {', '.join(f'{get_abundance(mol, i):.3g}' for i in isos)}) · "
          f"T mín {T.min():.4g} en {um[k]:.3f} µm · T media {T.mean():.4f} · "
          f"{time.perf_counter() - t0:.1f} s")
    if T.min() == 1.0:
        print(f"  Nota: τ máx = {float(tau.max()):.1e}; las líneas de {mol} en 8–14 µm son "
              "tan débiles que e^(−τ) = 1 en doble precisión (no es un error).")
    RESULTS.append((mol, "OK", f"{sd.n_lines:,} líneas, T mín {T.min():.4g} en {um[k]:.3f} µm"))


def _fail(mol, err, why, t0):
    print(f"✗ {mol}: {type(err).__name__}")
    print(f"  Por qué: {why}")
    print(f"  Mensaje: {err}")
    RESULTS.append((mol, type(err).__name__, why))

## 1. H2O (mol_id 1)

In [ ]:
firma("H2O")

## 2. CO2 (mol_id 2)

In [ ]:
firma("CO2")

## 3. O3 (mol_id 3)

In [ ]:
firma("O3")

## 4. N2O (mol_id 4)

In [ ]:
firma("N2O")

## 5. CO (mol_id 5)

In [ ]:
firma("CO")

## 6. CH4 (mol_id 6)

In [ ]:
firma("CH4")

## 7. O2 (mol_id 7)

In [ ]:
firma("O2")

## 8. NO (mol_id 8)

In [ ]:
firma("NO")

## 9. SO2 (mol_id 9)

In [ ]:
firma("SO2")

## 10. NO2 (mol_id 10)

In [ ]:
firma("NO2")

## 11. NH3 (mol_id 11)

In [ ]:
firma("NH3")

## 12. HNO3 (mol_id 12)

In [ ]:
firma("HNO3")

## 13. OH (mol_id 13)

In [ ]:
firma("OH")

## 14. HF (mol_id 14)

In [ ]:
firma("HF")

## 15. HCl (mol_id 15)

In [ ]:
firma("HCl")

## 16. HBr (mol_id 16)

In [ ]:
firma("HBr")

## 17. HI (mol_id 17)

In [ ]:
firma("HI")

## 18. ClO (mol_id 18)

In [ ]:
firma("ClO")

## 19. OCS (mol_id 19)

In [ ]:
firma("OCS")

## 20. H2CO (mol_id 20)

In [ ]:
firma("H2CO")

## 21. HOCl (mol_id 21)

In [ ]:
firma("HOCl")

## 22. N2 (mol_id 22)

In [ ]:
firma("N2")

## 23. HCN (mol_id 23)

In [ ]:
firma("HCN")

## 24. CH3Cl (mol_id 24)

In [ ]:
firma("CH3Cl")

## 25. H2O2 (mol_id 25)

In [ ]:
firma("H2O2")

## 26. C2H2 (mol_id 26)

In [ ]:
firma("C2H2")

## 27. C2H6 (mol_id 27)

In [ ]:
firma("C2H6")

## 28. PH3 (mol_id 28)

In [ ]:
firma("PH3")

## 29. COF2 (mol_id 29)

In [ ]:
firma("COF2")

## 30. SF6 (mol_id 30)

In [ ]:
firma("SF6")

## 31. H2S (mol_id 31)

In [ ]:
firma("H2S")

## 32. HCOOH (mol_id 32)

In [ ]:
firma("HCOOH")

## 33. HO2 (mol_id 33)

In [ ]:
firma("HO2")

## 34. O (mol_id 34)

In [ ]:
firma("O")

## 35. ClONO2 (mol_id 35)

In [ ]:
firma("ClONO2")

## 36. NO+ (mol_id 36)

In [ ]:
firma("NO+")

## 37. HOBr (mol_id 37)

In [ ]:
firma("HOBr")

## 38. C2H4 (mol_id 38)

In [ ]:
firma("C2H4")

## 39. CH3OH (mol_id 39)

In [ ]:
firma("CH3OH")

## 40. CH3Br (mol_id 40)

In [ ]:
firma("CH3Br")

## 41. CH3CN (mol_id 41)

In [ ]:
firma("CH3CN")

## 42. CF4 (mol_id 42)

In [ ]:
firma("CF4")

## 43. C4H2 (mol_id 43)

In [ ]:
firma("C4H2")

## 44. HC3N (mol_id 44)

In [ ]:
firma("HC3N")

## 45. H2 (mol_id 45)

In [ ]:
firma("H2")

## 46. CS (mol_id 46)

In [ ]:
firma("CS")

## 47. SO3 (mol_id 47)

In [ ]:
firma("SO3")

## 48. C2N2 (mol_id 48)

In [ ]:
firma("C2N2")

## 49. COCl2 (mol_id 49)

In [ ]:
firma("COCl2")

## 50. SO (mol_id 50)

In [ ]:
firma("SO")

## 51. CH3F (mol_id 51)

In [ ]:
firma("CH3F")

## 52. GeH4 (mol_id 52)

In [ ]:
firma("GeH4")

## 53. CS2 (mol_id 53)

In [ ]:
firma("CS2")

## 54. CH3I (mol_id 54)

In [ ]:
firma("CH3I")

## 55. NF3 (mol_id 55)

In [ ]:
firma("NF3")

## 56. H3+ (mol_id 56)

In [ ]:
firma("H3+")

## 57. CH3 (mol_id 57)

In [ ]:
firma("CH3")

## 58. S2 (mol_id 58)

In [ ]:
firma("S2")

## 59. COFCl (mol_id 59)

In [ ]:
firma("COFCl")

## 60. HONO (mol_id 60)

In [ ]:
firma("HONO")

## 61. ClNO2 (mol_id 61)

In [ ]:
firma("ClNO2")

## Resumen

In [ ]:
ok = [r for r in RESULTS if r[1] == "OK"]
print(f"{len(ok)} de {len(RESULTS)} moléculas con firma en 8–14 µm\n")
print(f"{'Molécula':<8}  {'Resultado':<24}  Detalle")
print("─" * 100)
for mol, status, detail in RESULTS:
    print(f"{mol:<8}  {status:<24}  {detail}")